# Solar Domain Baseline — LSTM



## 1. Setup

In [1]:
from pathlib import Path
import json
import random
import sys

import numpy as np
import pandas as pd

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)

set_seed()

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "requirements").exists() and (candidate / "solar").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Run this notebook from inside the cloned repo."
    )

REPO_ROOT = find_repo_root()
ROOT = REPO_ROOT / "solar"
DATA_PATH = ROOT / "data" / "solardata_modified.csv"
PRED_DIR = ROOT / "outputs" / "predictions"
METRIC_DIR = ROOT / "outputs" / "metrics"

PRED_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Solar root:", ROOT)
print("Data:", DATA_PATH)

# Shared solar utilities
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from src.metrics import compute_metrics, print_metrics
from power_conversion import convert_solar_result_to_power


import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import MinMaxScaler

def set_torch_seed(seed=SEED):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_torch_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [4]:
TEST_START = pd.Timestamp("2023-10-01")
DAY_HOURS = list(range(8, 19))
SEQ_LEN = len(DAY_HOURS)

MAIN_QUANTILES = [round(q, 1) for q in np.arange(0.1, 1.0, 0.1)]
TAIL_QUANTILES = [0.01, 0.025, 0.05, 0.95, 0.975, 0.99]
QUANTILES = sorted(set(MAIN_QUANTILES + TAIL_QUANTILES))

BATCH_SIZE = 50
HIDDEN_SIZE = 30
EPOCHS = 50
LEARNING_RATE = 1e-3

OUTPUT_PATH = PRED_DIR / "lstm_quantile.csv"
METRICS_PATH = METRIC_DIR / "lstm_quantile.json"

## 2. Load data

In [2]:
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values(["item_id", "timestamp"]).reset_index(drop=True)

print(df.shape)
df.head()

## 3. Preprocessing

In [6]:
WEATHER_LAG24 = [
    "AIRTEMP_lag24", "RH_AVG_lag24", "DEWPT_lag24",
    "WS_lag24", "GHI_lag24", "DNI_lag24", "DIFF_lag24",
]

FEATURE_COLS = [
    "month", "day", "hour",
    *WEATHER_LAG24,
    "POA_lag24", "POA_lag48", "POA_lag168",
]

missing = [c for c in WEATHER_LAG24 if c not in df.columns]
if missing:
    raise KeyError(f"Missing expected lag-24 weather columns: {missing}")


def preprocess(df):
    out = df.copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"])
    out = out.sort_values(["item_id", "timestamp"]).copy()

    for lag in (24, 48, 168):
        out[f"POA_lag{lag}"] = out.groupby("item_id")["target"].shift(lag)

    out["date"] = out["timestamp"].dt.normalize()
    out["month"] = out["timestamp"].dt.month
    out["day"] = out["timestamp"].dt.day
    out["hour"] = out["timestamp"].dt.hour

    return out[out["hour"].isin(DAY_HOURS)].copy()


def build_sequences(day_df):
    X_list, y_list, meta_list, sequence_dates = [], [], [], []

    for item_id, item_df in day_df.groupby("item_id", sort=False):
        for date, day in item_df.groupby("date", sort=True):
            day = day.sort_values("hour")

            if day["hour"].tolist() != DAY_HOURS:
                continue
            if day[FEATURE_COLS + ["target"]].isna().any().any():
                continue

            X_list.append(day[FEATURE_COLS].to_numpy(dtype=np.float32))
            y_list.append(day["target"].to_numpy(dtype=np.float32))
            meta_list.append(day[["timestamp", "item_id"]].reset_index(drop=True))
            sequence_dates.append(pd.Timestamp(date))

    if not X_list:
        raise ValueError("No complete daylight sequences were created.")

    return (
        np.stack(X_list),
        np.stack(y_list),
        meta_list,
        np.asarray(sequence_dates, dtype="datetime64[ns]"),
    )


day_df = preprocess(df)
X, y, meta, sequence_dates = build_sequences(day_df)

print("X:", X.shape)
print("y:", y.shape)

X: (723, 11, 13)
y: (723, 11)


## 4. split and standization

In [3]:
train_mask = pd.to_datetime(sequence_dates) < TEST_START
test_mask = ~train_mask

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]
meta_test = [m for m, keep in zip(meta, test_mask) if keep]

n_features = X_train.shape[-1]

x_scaler = MinMaxScaler(feature_range=(-1, 1))
y_scaler = MinMaxScaler(feature_range=(-1, 1))

X_train_s = x_scaler.fit_transform(
    X_train.reshape(-1, n_features)
).reshape(X_train.shape)

X_test_s = x_scaler.transform(
    X_test.reshape(-1, n_features)
).reshape(X_test.shape)

y_train_s = y_scaler.fit_transform(
    y_train.reshape(-1, 1)
).reshape(y_train.shape)

print("Training sequences:", len(X_train_s))
print("Test sequences:", len(X_test_s))

## 5. LSTM model

In [8]:
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class LSTMQuantile(nn.Module):
    def __init__(self, input_size, hidden_size, quantiles):
        super().__init__()
        self.quantiles = list(quantiles)
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, len(self.quantiles))

    def forward(self, x):
        h, _ = self.lstm(x)
        return self.head(h)


def quantile_loss(pred, target, quantiles):
    losses = []
    for idx, q in enumerate(quantiles):
        error = target - pred[:, :, idx]
        losses.append(torch.maximum(q * error, (q - 1) * error).mean())
    return torch.stack(losses).mean()

## 6. Train

In [4]:
train_loader = DataLoader(
    SequenceDataset(X_train_s, y_train_s),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

model = LSTMQuantile(
    input_size=n_features,
    hidden_size=HIDDEN_SIZE,
    quantiles=QUANTILES,
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()
        pred = model(X_batch)
        loss = quantile_loss(pred, y_batch, QUANTILES)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    if epoch == 0 or (epoch + 1) % 10 == 0:
        mean_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch + 1:03d}/{EPOCHS}: loss={mean_loss:.6f}")

## 7. Test

In [5]:
model.eval()
with torch.no_grad():
    pred_s = model(
        torch.tensor(X_test_s, dtype=torch.float32, device=DEVICE)
    ).cpu().numpy()

B, T, Q = pred_s.shape
pred = np.empty_like(pred_s)

for q_idx in range(Q):
    pred[:, :, q_idx] = y_scaler.inverse_transform(
        pred_s[:, :, q_idx].reshape(-1, 1)
    ).reshape(B, T)

# prevent crossing
pred = np.sort(pred, axis=2)
pred = np.clip(pred, a_min=0.0, a_max=None)

print(pred.shape)

## 8. convert to 24 hour output

In [6]:
def build_daylight_output(pred, y_test, meta_test, quantiles):
    rows = []
    for i, block_meta in enumerate(meta_test):
        block = block_meta.copy()
        block["target"] = y_test[i]

        for q_idx, q in enumerate(quantiles):
            block[str(q)] = pred[i, :, q_idx]

        rows.append(block)

    return pd.concat(rows, ignore_index=True)


def restore_full_test_grid(full_df, daylight_output, test_start, quantiles):
    full = full_df.copy()
    full["timestamp"] = pd.to_datetime(full["timestamp"])
    full = full[full["timestamp"] >= test_start][
        ["timestamp", "item_id", "target"]
    ].copy()

    q_cols = [str(q) for q in quantiles]
    out = full.merge(
        daylight_output[["timestamp", "item_id", *q_cols]],
        on=["timestamp", "item_id"],
        how="left",
    )

    for col in q_cols:
        out[col] = out[col].fillna(0.0).clip(lower=0.0)

    out["prediction"] = out["0.5"]
    return out.sort_values(["item_id", "timestamp"]).reset_index(drop=True)


daylight_output = build_daylight_output(pred, y_test, meta_test, QUANTILES)
final_df = restore_full_test_grid(df, daylight_output, TEST_START, QUANTILES)

final_df.head()

## 9. Evaluation


In [7]:

poa_metrics = print_metrics(
    final_df,
    quantiles=MAIN_QUANTILES,
    label="Solar POA",
)

# power conversion
power_df = convert_solar_result_to_power(
    final_df,
    df,
    temperature_col="AIRTEMP",
)

power_metrics = print_metrics(
    power_df,
    quantiles=MAIN_QUANTILES,
    label="Solar power",
)


## 10. Save

In [14]:

final_df.to_csv(OUTPUT_PATH, index=False)

